In [ ]:
import os
from pathlib import Path

import pandas as pd
from src import analysis
from src.analysis import evaluation

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Use CPU for evaluation

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
# Models / architectures to compare
MODELS = [
    # # --- FNO ---
    # # FNO small
    # "FNO_m12x12_h24_l4_lhs_var80_seed3001_20260109_201346"
    # # FNO big
    "FNO_m48x48_h64_l4_lhs_var80_seed3001_20260110_085608",
    # # --- PI-FNO ---
    # # PI-FNO small
    "PI-FNO_m12x12_h24_l4_lamPhys1e-04_lamP5e-03_lhs_var80_seed3001_20260109_191704",
    # # PI-FNO big
    "PI-FNO_m48x48_h64_l4_lamPhys1e-04_lamP5e-04_lhs_var80_seed3001_20260110_225103",
    # --- U-NO ---
    # U-NO small
    # U-NO big
    # --- PI-U-NO ---
    # PI-U-NO small
    # PI-U-NO big
]

MODEL_LABELS = {
    "FNO_m48x48_h64_l4_lhs_var80_seed3001_20260110_085608": "FNO_m48x48_h64_l4",
    "PI-FNO_m12x12_h24_l4_lamPhys1e-04_lamP5e-03_lhs_var80_seed3001_20260109_191704": "PI-FNO_m12x12_h24_l4_lamPhys1e-04_lamP5e-03",
    "PI-FNO_m48x48_h64_l4_lamPhys1e-04_lamP5e-04_lhs_var80_seed3001_20260110_225103": "PI-FNO_m48x48_h64_l4_lamPhys1e-04_lamP5e-04",
}

# Shared datasets (IDENTICAL for all models)
ID_DATASET = "lhs_var80_seed3001"
OOD_DATASET = "lhs_var120_seed4001"

show_id_evaluation = True
show_ood_evaluation = False

In [ ]:
# ======================================================================
# Paths
# ======================================================================
RAW_ROOT = Path("../../data/raw")
PROCESSED_ROOT = Path("../data/processed")


def run_or_load_artifacts_evaluation(
    *,
    model_name: str,
    dataset_name: str,
) -> pd.DataFrame:
    """Load or generate evaluation artifacts for one model on one dataset."""
    run_dir = PROCESSED_ROOT / model_name
    checkpoint_path = run_dir / "best_model_state_dict.pt"

    dataset_path = RAW_ROOT / dataset_name / "cases"

    save_root = run_dir / "analysis" / "id" if dataset_name == ID_DATASET else run_dir / "analysis" / "ood" / dataset_name

    parquet_path = save_root / f"{dataset_name}.parquet"

    if parquet_path.exists():
        print(f"[INFO] Loading {model_name} | {dataset_name}")
        return pd.read_parquet(parquet_path)

    print(f"[INFO] Creating artifacts: {model_name} | {dataset_name}")

    model, loader, processor, device = analysis.analysis_interference.load_inference_context(
        dataset_path=dataset_path,
        checkpoint_path=checkpoint_path,
        batch_size=1,
    )

    df, _ = analysis.analysis_artifacts.generate_artifacts(
        model=model,
        loader=loader,
        processor=processor,
        device=device,
        save_root=save_root,
        dataset_name=dataset_name,
    )

    return df


def make_model_label(model_name: str, dataset_name: str) -> str:
    """Remove dataset name and trailing timestamp from model name."""
    token = f"_{dataset_name}_"
    if token in model_name:
        return model_name.split(token)[0]
    return model_name

In [ ]:
# ======================================================================
# Load evaluation data
# ======================================================================
datasets_eval_id = {}
datasets_eval_ood = {}

for model_name in MODELS:
    print(f"\n=== {model_name} ===")

    if show_id_evaluation:
        # -----------------
        # ID
        # -----------------
        df_raw_id = run_or_load_artifacts_evaluation(
            model_name=model_name,
            dataset_name=ID_DATASET,
        )
        df_id = evaluation.evaluation_dataframe.build_eval_df(df_raw_id)
        label = make_model_label(model_name, ID_DATASET)
        datasets_eval_id[label] = df_id

    if show_ood_evaluation:
        # -----------------
        # OOD
        # -----------------
        df_raw_ood = run_or_load_artifacts_evaluation(
            model_name=model_name,
            dataset_name=OOD_DATASET,
        )
        df_ood = evaluation.evaluation_dataframe.build_eval_df(df_raw_ood)
        label = make_model_label(model_name, ID_DATASET)
        datasets_eval_ood[label] = df_ood

In [ ]:
if show_id_evaluation:
    panel_id = evaluation.evaluation_panel.build_evaluation_panel(
        datasets_eval=datasets_eval_id,
        title="Model Comparison ID",
        sections="all",
    )
    display(panel_id)

In [ ]:
if show_ood_evaluation:
    panel_ood = evaluation.evaluation_panel.build_evaluation_panel(
        datasets_eval=datasets_eval_ood,
        title="Model Comparison OOD",
        sections="all",
    )
    display(panel_ood)